# Mistral Agent on Lunar Sandbox

This notebook runs a Mistral AI agent inside an isolated Docker sandbox.
Every action is auto-traced to the dashboard at http://localhost:3000.

Mistral's API is OpenAI-compatible, so we use the `openai` SDK pointed at Mistral's endpoint.

In [1]:
# Install the OpenAI SDK (run once)
!pip install openai -q

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

# Ensure docker CLI is on PATH (Jupyter kernels may not inherit the full shell PATH)
os.environ["PATH"] = "/usr/local/bin:/opt/homebrew/bin:" + os.environ.get("PATH", "")

os.environ["MISTRAL_API_KEY"] = ""  # <-- set your Mistral API key

## Example 1 — Agent loop

One cell does everything: creates a sandbox, gives the agent a task, and traces every step to the dashboard.

In [3]:
from openai import OpenAI
from lunar_sandbox import Session, openai_adapter

client = OpenAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    base_url="https://api.mistral.ai/v1",
)

with Session("mistral-demo", image="python:3.12-slim") as s:
    # Install pytest in the sandbox
    s.run("pip install -q pytest")

    # Let the agent write code and tests
    answer = s.agent_loop(
        task=(
            "Create a Python script called weather.py that:\n"
            "1. Defines a dict mapping 5 cities to their temperatures in Celsius\n"
            "2. Finds the hottest and coldest city\n"
            "3. Prints a formatted summary\n"
            "4. Then create test_weather.py with pytest tests and run them"
        ),
        call_llm=openai_adapter(client, model="mistral-small-latest"),
        max_steps=20,
    )
    print(f"Agent: {answer}")

    # Verify
    print("\n=== Files ===")
    print(s.list_files())
    print("\n=== weather.py ===")
    print(s.read_file("weather.py"))
    print("\n=== Test results ===")
    print(s.run("cd /workspace && python -m pytest test_weather.py -v"))

    s.finish(score=1.0)

2026-03-22 15:48:20 [debug    ] seccomp_module_not_available  
2026-03-22 15:48:20 [debug    ] trajectory_store_opened        db_path=/Users/diogovieira/Developer/sandbox/trajectories/trajectories.db
2026-03-22 15:48:20 [info     ] episode_ingested               episode_id=ep-348eb87682db step_count=0
2026-03-22 15:48:20 [debug    ] docker_sandbox_initialized     sandbox_id=session-80a01d66cbfd state=created
2026-03-22 15:48:20 [info     ] docker_sandbox_creating        sandbox_id=session-80a01d66cbfd
2026-03-22 15:48:21 [debug    ] health_mount_recorded          mount_count=1 sandbox_id=session-80a01d66cbfd
2026-03-22 15:48:21 [info     ] docker_sandbox_created         container_id=76ab8712601b image=python:3.12-slim sandbox_id=session-80a01d66cbfd
2026-03-22 15:48:21 [info     ] session_started                image=python:3.12-slim session_episode=ep-348eb87682db session_sandbox=session-80a01d66cbfd task=mistral-demo url=http://localhost:3000/runs/ep-348eb87682db
Session started: htt

## Example 2 — Manual tool loop

For full control, use `s.tools()` and `s.call_tool()` directly.

In [ ]:
import json
from lunar_sandbox import Session

client = OpenAI(
    api_key=os.environ["MISTRAL_API_KEY"],
    base_url="https://api.mistral.ai/v1",
)

with Session("mistral-manual", image="python:3.12-slim") as s:
    tools = s.tools(format="openai")
    messages = [
        {"role": "system", "content": "You are a coding assistant. Use the tools to write and run code in /workspace."},
        {"role": "user", "content": "Create a fibonacci function in fib.py, then run it to print the first 10 numbers."},
    ]

    for step in range(15):
        resp = client.chat.completions.create(
            model="mistral-small-latest",
            messages=messages,
            tools=tools,
        )
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"Agent: {msg.content}")
            break

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = s.call_tool(tc.function.name, args)
            print(f"  [{tc.function.name}] -> {result[:200]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    s.finish(score=1.0)